# CICIoMT2024 — Kaggle Intrusion Detection Model

هذا الـNotebook يبني نموذج كشف وتصنيف هجمات على ملفات **WiFI_and_MQTT** الجاهزة بصيغة CSV.

- الوضع الافتراضي: تصنيف متعدد الفئات (Multiclass).
- لتجربة كشف ثنائي Benign/Attack غيّر `TASK` إلى `binary`.
- يتم اشتقاق الفئة من اسم الملف، لأن كل ملف يمثل نوع traffic محدد.
- يتم توحيد الملفات المجزأة مثل `ICMP1 ... ICMP8` داخل فئة `ICMP` واحدة.
- نستخدم عينات موزونة ومتوازنة لكل فئة حتى يعمل ضمن RAM ووقت Kaggle.


In [ ]:
from pathlib import Path
from collections import defaultdict
import math
import re
import time
import warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
SEED = 42
np.random.seed(SEED)


## 1) الإعدادات ومسارات Kaggle

إذا تغيّر اسم مجلد الداتا في Kaggle، الكود يبحث عنه تلقائيًا داخل `/kaggle/input`.


In [ ]:
# اختر: multiclass أو binary
TASK = "multiclass"

# حدود تحفظ الذاكرة والوقت. ارفعها لاحقًا إذا كانت الجلسة تسمح.
MAX_TRAIN_ROWS_PER_CLASS = 40_000
MAX_TEST_ROWS_PER_CLASS = 10_000
CSV_CHUNK_SIZE = 100_000

def find_dataset_root():
    preferred = Path("/kaggle/input/ciciomt2024/Dataset/WiFI_and_MQTT")
    if preferred.exists():
        return preferred

    candidates = list(Path("/kaggle/input").glob("**/WiFI_and_MQTT"))
    if not candidates:
        raise FileNotFoundError(
            "لم أجد مجلد WiFI_and_MQTT. أضف Dataset CICIoMT2024 إلى Input ثم أعد التشغيل."
        )
    return candidates[0]

DATA_ROOT = find_dataset_root()
TRAIN_DIR = DATA_ROOT / "train"
TEST_DIR = DATA_ROOT / "test"

train_files = sorted(TRAIN_DIR.glob("*.csv"))
test_files = sorted(TEST_DIR.glob("*.csv"))

assert train_files, f"لا توجد ملفات CSV في {TRAIN_DIR}"
assert test_files, f"لا توجد ملفات CSV في {TEST_DIR}"

print("Data root:", DATA_ROOT)
print("Train files:", len(train_files))
print("Test files :", len(test_files))
print("First train file:", train_files[0].name)


## 2) بناء الـlabels وتحديد الأعمدة المشتركة

مثال: `TCP_IP-DDoS-ICMP7_train.pcap.csv` يصبح `TCP_IP-DDoS-ICMP`.


In [ ]:
def attack_name_from_file(path):
    name = re.sub(
        r"_(train|test)\.pcap\.csv$",
        "",
        Path(path).name,
        flags=re.IGNORECASE,
    )
    # دمج shards: ICMP1..8, SYN1..4, TCP1..4, UDP1..3
    name = re.sub(r"(?<=[A-Za-z])\d+$", "", name)
    return name

def target_from_file(path, task=TASK):
    attack = attack_name_from_file(path)
    if task == "binary":
        return "Benign" if attack.lower() == "benign" else "Attack"
    if task != "multiclass":
        raise ValueError("TASK يجب أن يكون multiclass أو binary")
    return attack

# نأخذ تقاطع الأعمدة بين كل ملفات train/test وبنفس ترتيب الملف الأول.
first_columns = list(pd.read_csv(train_files[0], nrows=0).columns)
common_columns = set(first_columns)
for path in train_files + test_files:
    common_columns &= set(pd.read_csv(path, nrows=0).columns)

label_like = {"label", "class", "target", "category", "attack", "type"}
feature_columns = [
    col for col in first_columns
    if col in common_columns and col.strip().lower() not in label_like
]

print("Common feature columns:", len(feature_columns))
print("Train classes:")
print(pd.Series([target_from_file(p) for p in train_files]).value_counts().sort_index())

unknown_test_classes = set(target_from_file(p) for p in test_files) - set(
    target_from_file(p) for p in train_files
)
assert not unknown_test_classes, f"فئات موجودة في test وليست في train: {unknown_test_classes}"


## 3) تحميل عينة متوازنة بذاكرة محدودة

القراءة تتم على chunks. لكل صف رقم عشوائي، ونحتفظ بأصغر الأرقام؛ هذه عينة عشوائية موزعة على كامل الملف بدل أخذ أول الصفوف فقط.


In [ ]:
def reservoir_sample_csv(path, n_rows, usecols, seed, chunksize=CSV_CHUNK_SIZE):
    rng = np.random.default_rng(seed)
    kept = None

    for chunk in pd.read_csv(path, usecols=usecols, chunksize=chunksize):
        chunk = chunk.apply(pd.to_numeric, errors="coerce")
        chunk = chunk.replace([np.inf, -np.inf], np.nan)
        chunk["__priority__"] = rng.random(len(chunk))

        kept = chunk if kept is None else pd.concat([kept, chunk], ignore_index=True)
        if len(kept) > n_rows + chunksize:
            kept = kept.nsmallest(n_rows, "__priority__")

    if kept is None:
        return pd.DataFrame(columns=usecols)

    return (
        kept.nsmallest(min(n_rows, len(kept)), "__priority__")
        .drop(columns="__priority__")
        .reset_index(drop=True)
    )

def load_balanced_split(files, max_rows_per_class, split_seed):
    grouped = defaultdict(list)
    for path in files:
        grouped[target_from_file(path)].append(path)

    class_frames = []
    for class_index, (label, paths) in enumerate(sorted(grouped.items())):
        per_file = math.ceil(max_rows_per_class / len(paths))
        parts = []

        for file_index, path in enumerate(paths):
            sample_seed = split_seed + class_index * 1_000 + file_index
            part = reservoir_sample_csv(
                path,
                n_rows=per_file,
                usecols=feature_columns,
                seed=sample_seed,
            )
            parts.append(part)

        class_df = pd.concat(parts, ignore_index=True)
        if len(class_df) > max_rows_per_class:
            class_df = class_df.sample(
                n=max_rows_per_class, random_state=split_seed + class_index
            )

        class_df["__target__"] = label
        class_frames.append(class_df)
        print(f"{label:35s} -> {len(class_df):,} rows from {len(paths)} file(s)")

    result = pd.concat(class_frames, ignore_index=True)
    return result.sample(frac=1, random_state=split_seed).reset_index(drop=True)

start = time.time()
train_df = load_balanced_split(train_files, MAX_TRAIN_ROWS_PER_CLASS, SEED)
test_df = load_balanced_split(test_files, MAX_TEST_ROWS_PER_CLASS, SEED + 10_000)

print(f"\nLoaded in {(time.time() - start) / 60:.1f} minutes")
print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)
display(train_df["__target__"].value_counts().sort_index().to_frame("train_rows"))


## 4) تنظيف أخير وتدريب HistGradientBoosting

هذا النموذج مناسب للخصائص الرقمية، لا يحتاج StandardScaler، ويتعامل مع القيم المفقودة. نستخدم أوزان فئات إضافية إذا بقي اختلاف في عدد الصفوف.


In [ ]:
# حذف الأعمدة التي لا تحمل معلومات في عينة التدريب.
usable_features = [
    col for col in feature_columns
    if train_df[col].notna().any() and train_df[col].nunique(dropna=True) > 1
]
dropped_features = sorted(set(feature_columns) - set(usable_features))
print("Usable features:", len(usable_features))
print("Dropped constant/empty features:", dropped_features)

X_train = train_df[usable_features].astype("float32")
X_test = test_df[usable_features].astype("float32")

label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(train_df["__target__"])
y_test = label_encoder.transform(test_df["__target__"])
sample_weight = compute_sample_weight(class_weight="balanced", y=y_train)

model = HistGradientBoostingClassifier(
    learning_rate=0.08,
    max_iter=160,
    max_leaf_nodes=31,
    min_samples_leaf=30,
    l2_regularization=1.0,
    early_stopping=True,
    validation_fraction=0.10,
    n_iter_no_change=12,
    random_state=SEED,
)

start = time.time()
model.fit(X_train, y_train, sample_weight=sample_weight)
print(f"Training time: {(time.time() - start) / 60:.1f} minutes")
print("Boosting iterations:", model.n_iter_)


## 5) التقييم على ملفات test الأصلية

لا تعتمد على Accuracy وحدها. الأهم هنا `Macro F1` و`Balanced Accuracy` بسبب اختلاف أحجام الفئات.


In [ ]:
y_pred = model.predict(X_test)

macro_f1 = f1_score(y_test, y_pred, average="macro")
weighted_f1 = f1_score(y_test, y_pred, average="weighted")
balanced_acc = balanced_accuracy_score(y_test, y_pred)

print(f"Macro F1         : {macro_f1:.4f}")
print(f"Weighted F1      : {weighted_f1:.4f}")
print(f"Balanced Accuracy: {balanced_acc:.4f}")
print()
print(classification_report(
    y_test,
    y_pred,
    target_names=label_encoder.classes_,
    digits=4,
    zero_division=0,
))

cm = confusion_matrix(y_test, y_pred, normalize="true")
fig_size = max(9, len(label_encoder.classes_) * 0.65)
plt.figure(figsize=(fig_size, fig_size))
sns.heatmap(
    cm,
    cmap="Blues",
    vmin=0,
    vmax=1,
    xticklabels=label_encoder.classes_,
    yticklabels=label_encoder.classes_,
    annot=(len(label_encoder.classes_) <= 16),
    fmt=".2f",
    cbar_kws={"label": "Recall per true class"},
)
plt.title(f"CICIoMT2024 — Normalized Confusion Matrix ({TASK})")
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.xticks(rotation=75, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


## 6) حفظ النموذج

سيظهر الملف في قسم Output بعد تشغيل **Save Version** على Kaggle.


In [ ]:
artifact = {
    "model": model,
    "label_encoder": label_encoder,
    "feature_columns": usable_features,
    "task": TASK,
    "dataset": "CICIoMT2024 / WiFI_and_MQTT",
    "metrics": {
        "macro_f1": float(macro_f1),
        "weighted_f1": float(weighted_f1),
        "balanced_accuracy": float(balanced_acc),
    },
}

model_path = Path("/kaggle/working") / f"ciciomt2024_{TASK}_hgb.joblib"
joblib.dump(artifact, model_path)
print(f"Saved: {model_path} ({model_path.stat().st_size / 1e6:.1f} MB)")


## تحسينات لاحقة مقترحة

1. شغّل `binary` أولًا للتأكد من الـpipeline.
2. شغّل `multiclass` وراقب الفئات الضعيفة في confusion matrix.
3. ارفع عدد الصفوف لكل فئة تدريجيًا إن كانت الذاكرة والمدة تسمحان.
4. قارن HistGradientBoosting مع LightGBM/XGBoost بنفس train/test الرسميين.
5. Bluetooth ليس CSV؛ يحتاج استخراج features من PCAP (مثل packet length, timing, direction, protocol flags) في Notebook مستقل.
